# 02 — Clean and transform

Run after 01, in a fresh kernel. The actual transformation code is in this notebook. Work through each definition, then run the execution cells. Outputs replace the prior generated data; the raw file is never changed.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "PROBLEM_STATEMENT.md").is_file())
RAW = ROOT / "data/raw/online_retail_II.csv"
PROCESSED = ROOT / "data/processed"
QUALITY = ROOT / "data/quality"
TABLES = ROOT / "power_bi/tables"
SOURCE_COLUMNS = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "CustomerID", "Country"]
FACT_COLUMNS = ["SourceRow", *SOURCE_COLUMNS, "InvoiceDay", "RevenuePence", "TotalAmount", "Year", "Month", "MonthName", "DayOfWeek", "WeekdayNumber", "Hour"]
SOURCE = RAW  # For the original workbook, use ROOT / "data/raw/online_retail_II.xlsx".

## 1. Read and record the source

Keep source fields as text until explicit validation. SourceRow will preserve the original one-based data row.

In [ ]:
def write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )

def load_raw(path: Path = RAW) -> pd.DataFrame:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Source not found: {path}. See README.md, Reproduce the analysis."
        )
    if path.suffix.lower() == ".xlsx":
        # UCI's workbook has two annual sheets; reading only the first loses a year.
        frame = pd.concat(
            pd.read_excel(path, sheet_name=None, dtype="string").values(),
            ignore_index=True,
        )
    else:
        frame = pd.read_csv(path, dtype="string")
    frame.columns = frame.columns.str.strip().str.replace(" ", "", regex=False)
    frame = frame.rename(columns={"InvoiceNo": "Invoice", "UnitPrice": "Price"})
    missing = set(SOURCE_COLUMNS) - set(frame.columns)
    if missing or frame.columns.duplicated().any():
        raise ValueError(
            f"Invalid source columns; missing={sorted(missing)} or duplicate headers."
        )
    return frame[SOURCE_COLUMNS].copy()

## 2. Define the eligible population

Each rejected row gets only its first reason. The order matters: customer → cancellation → keys → date → quantity → price → repeated fields. Positive unit prices retain up to three decimals; line totals round half up to integer pence. Missing descriptions become Unknown Product.

In [ ]:
def clean(frame: pd.DataFrame) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    """Assign one rejection reason per row, preserving 1-based source row lineage."""
    data = frame[SOURCE_COLUMNS].copy().reset_index(drop=True)
    for column in SOURCE_COLUMNS:
        data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
    reason = pd.Series("", index=data.index, dtype="string")

    def reject(mask: pd.Series, label: str) -> None:
        reason.loc[reason.eq("") & mask.fillna(True)] = label

    customer = pd.to_numeric(data.CustomerID, errors="coerce")
    quantity = pd.to_numeric(data.Quantity, errors="coerce")
    price = pd.to_numeric(data.Price, errors="coerce")
    dates = pd.to_datetime(data.InvoiceDate, errors="coerce", format="mixed")
    reject(data.CustomerID.isna(), "missing_customer")
    reject(
        customer.isna()
        | ~np.isfinite(customer)
        | customer.le(0)
        | customer.mod(1).ne(0),
        "invalid_customer",
    )
    reject(
        data.Invoice.str.upper().str.startswith("C").fillna(False), "cancelled_invoice"
    )
    reject(
        data.Invoice.isna() | data.StockCode.isna() | data.Country.isna(),
        "missing_business_key",
    )
    reject(dates.isna(), "invalid_datetime")
    reject(
        quantity.isna()
        | ~np.isfinite(quantity)
        | quantity.le(0)
        | quantity.mod(1).ne(0)
        | quantity.gt(2**31 - 1),
        "invalid_quantity",
    )
    reject(price.isna() | ~np.isfinite(price) | price.le(0), "invalid_price")
    # The source includes sub-penny prices. Preserve 3 decimal places and round
    # line totals half up to pence, rather than rounding unit prices to pennies.
    reject(
        ((price * 1000) - (price * 1000).round()).abs().gt(0.000001)
        | price.gt(999999999),
        "unsupported_price_precision",
    )
    reject(data.duplicated(SOURCE_COLUMNS, keep="first"), "exact_duplicate")
    rejected = pd.DataFrame({"SourceRow": data.index + 1, "Reason": reason})
    rejected = rejected.loc[reason.ne("")].reset_index(drop=True)
    keep = reason.eq("")
    fact = data.loc[keep].copy()
    fact.insert(0, "SourceRow", fact.index + 1)
    fact["CustomerID"] = customer.loc[keep].astype("int64").astype("string")
    fact["Quantity"] = quantity.loc[keep].astype("int64")
    fact["Price"] = price.loc[keep].astype(float)
    fact["InvoiceDate"] = dates.loc[keep]
    fact["Description"] = fact.Description.fillna("Unknown Product")
    mills = (fact.Price * 1000).round().astype("int64")
    if any(int(q) * int(p) > 2**63 - 6 for q, p in zip(fact.Quantity, mills)):
        raise ValueError("Line revenue exceeds supported integer range.")
    fact["RevenuePence"] = (mills * fact.Quantity + 5) // 10
    fact["TotalAmount"] = fact.RevenuePence / 100
    fact["InvoiceDay"] = fact.InvoiceDate.dt.strftime("%Y-%m-%d")
    fact["Year"] = fact.InvoiceDate.dt.year
    fact["Month"] = fact.InvoiceDate.dt.month
    fact["MonthName"] = fact.InvoiceDate.dt.month_name()
    fact["DayOfWeek"] = fact.InvoiceDate.dt.day_name()
    fact["WeekdayNumber"] = fact.InvoiceDate.dt.dayofweek + 1
    fact["Hour"] = fact.InvoiceDate.dt.hour
    if fact.empty:
        raise ValueError("No eligible sales remain after cleaning; inspect the source.")
    # A source invoice is the order key; conflicting customer assignments must
    # not be silently resolved by selecting an arbitrary customer.
    if fact.groupby("Invoice").CustomerID.nunique().gt(1).any():
        raise ValueError(
            "An invoice belongs to multiple customers; investigate source keys."
        )
    quality = {
        "source_rows": len(data),
        "eligible_sales_lines": len(fact),
        "excluded_rows": len(rejected),
        "exclusions": reason[reason.ne("")].value_counts().sort_index().to_dict(),
        "duplicate_policy": "Remove repeated source fields after trimming whitespace; keep first. No source line ID proves these are errors.",
        "currency": "GBP",
        "money_policy": "Positive line quantity * unit price (up to 3dp), half-up to pence",
        "invoice_multiple_countries": int(
            fact.groupby("Invoice").Country.nunique().gt(1).sum()
        ),
        "customer_multiple_countries": int(
            fact.groupby("CustomerID").Country.nunique().gt(1).sum()
        ),
        "zero_pence_positive_sales_lines": int(fact.RevenuePence.eq(0).sum()),
    }
    assert (
        quality["source_rows"]
        == quality["eligible_sales_lines"] + quality["excluded_rows"]
    )
    return fact[FACT_COLUMNS].reset_index(drop=True), quality, rejected

## 3. Define the headline metrics

AOV is sales divided by distinct invoices, not an average of line amounts. Distinct customer and invoice counts are non-additive across groups.

In [ ]:
def kpis(fact: pd.DataFrame) -> dict:
    orders = fact.Invoice.nunique()
    revenue = int(fact.RevenuePence.sum())
    frequencies = fact.groupby("CustomerID").Invoice.nunique()
    return {
        "sales_lines": len(fact),
        "revenue_pence": revenue,
        "revenue_gbp": revenue / 100,
        "orders": orders,
        "customers": fact.CustomerID.nunique(),
        "products": fact.StockCode.nunique(),
        "countries": fact.Country.nunique(),
        "quantity": int(fact.Quantity.sum()),
        "average_order_value_gbp": revenue / 100 / orders,
        "repeat_customers": int(frequencies.gt(1).sum()),
        "repeat_customer_rate": float(frequencies.gt(1).mean()),
        "first_date": fact.InvoiceDay.min(),
        "last_date": fact.InvoiceDay.max(),
        "currency": "GBP",
    }

## 4. Score observed customers

Recency uses midnight after the final observed date. Frequency counts distinct invoices; monetary value sums pence. Average percentile ranks preserve ties. RFM labels are historical heuristics, not confirmed churn or customer acquisition.

In [ ]:
def score(values: pd.Series, reverse: bool = False) -> pd.Series:
    """Tie-preserving percentile quintiles, stable under row reorder and tiny data."""
    result = np.ceil(values.rank(method="average", pct=True) * 5).astype(int).clip(1, 5)
    return 6 - result if reverse else result

def customer_tables(fact: pd.DataFrame) -> pd.DataFrame:
    snapshot = fact.InvoiceDate.max().normalize() + pd.Timedelta(days=1)
    customers = (
        fact.groupby("CustomerID", sort=True)
        .agg(
            FirstPurchase=("InvoiceDate", "min"),
            LastPurchase=("InvoiceDate", "max"),
            Frequency=("Invoice", "nunique"),
            MonetaryPence=("RevenuePence", "sum"),
            TotalItems=("Quantity", "sum"),
        )
        .reset_index()
    )
    customers["SnapshotDate"] = snapshot.strftime("%Y-%m-%d")
    customers["Recency"] = (snapshot - customers.LastPurchase.dt.normalize()).dt.days
    customers["Monetary"] = customers.MonetaryPence / 100
    customers["R_Score"] = score(customers.Recency, reverse=True)
    customers["F_Score"] = score(customers.Frequency)
    customers["M_Score"] = score(customers.MonetaryPence)
    r, f, m = customers.R_Score, customers.F_Score, customers.M_Score
    customers["Segment"] = np.select(
        [
            (r >= 4) & (f >= 4) & (m >= 4),
            (r >= 3) & (f >= 3) & (m >= 3),
            (r >= 4) & (f <= 2),
            (r <= 2) & (f >= 3),
            (r <= 2) & (f <= 2),
        ],
        [
            "Champions",
            "Loyal Customers",
            "Recent Low Frequency",
            "At Risk",
            "Lost Customers",
        ],
        default="Regular",
    )
    customers["FrequencySegment"] = np.select(
        [
            customers.Frequency.eq(1),
            customers.Frequency.le(5),
            customers.Frequency.le(10),
        ],
        ["One-Time Buyer", "Occasional (2-5)", "Regular (6-10)"],
        default="Loyal (11+)",
    )
    return customers

## 5. Build the analytical tables

One stock code has one latest observed label. Complete-month growth requires a valid comparison. Cohorts use first observed purchase: future cells are absent and observed inactivity is zero. Calendar covers complete years so Power BI can use a continuous date table.

In [ ]:
def aggregate(fact: pd.DataFrame, by: str | list[str]) -> pd.DataFrame:
    result = (
        fact.groupby(by, sort=True)
        .agg(
            RevenuePence=("RevenuePence", "sum"),
            Orders=("Invoice", "nunique"),
            Customers=("CustomerID", "nunique"),
            Quantity=("Quantity", "sum"),
            SalesLines=("Invoice", "size"),
        )
        .reset_index()
    )
    result["Revenue"] = result.RevenuePence / 100
    result["AvgOrderValue"] = result.Revenue / result.Orders
    return result

def build_tables(fact: pd.DataFrame) -> dict[str, pd.DataFrame]:
    customers = customer_tables(fact)
    orders = (
        fact.groupby("Invoice")
        .agg(
            CustomerID=("CustomerID", "first"),
            InvoiceDate=("InvoiceDate", "min"),
            RevenuePence=("RevenuePence", "sum"),
            Quantity=("Quantity", "sum"),
            Lines=("Invoice", "size"),
        )
        .reset_index()
    )
    orders["Revenue"] = orders.RevenuePence / 100
    monthly = aggregate(
        fact.assign(MonthStart=fact.InvoiceDate.dt.to_period("M").dt.to_timestamp()),
        "MonthStart",
    )
    monthly["FullCalendarMonth"] = (
        monthly.MonthStart.ge(pd.Timestamp(fact.InvoiceDay.min()))
    ) & (
        monthly.MonthStart.add(pd.offsets.MonthEnd(0)).le(
            pd.Timestamp(fact.InvoiceDay.max())
        )
    )
    monthly["MoMGrowth"] = monthly.Revenue.pct_change()
    adjacent = (
        (monthly.MonthStart.dt.year * 12 + monthly.MonthStart.dt.month).diff().eq(1)
    )
    # Only adjacent full months are eligible for full-month growth comparisons.
    monthly.loc[
        ~(
            adjacent
            & monthly.FullCalendarMonth
            & monthly.FullCalendarMonth.shift(1, fill_value=False)
        ),
        "MoMGrowth",
    ] = np.nan
    previous = monthly[["MonthStart", "Revenue", "FullCalendarMonth"]].copy()
    previous.MonthStart += pd.DateOffset(years=1)
    previous.columns = ["MonthStart", "PriorYearRevenue", "PriorYearFullMonth"]
    monthly = monthly.merge(previous, how="left", on="MonthStart")
    monthly["YoYGrowth"] = monthly.Revenue / monthly.PriorYearRevenue - 1
    monthly.loc[
        ~(monthly.FullCalendarMonth & monthly.PriorYearFullMonth.eq(True)), "YoYGrowth"
    ] = np.nan
    country = aggregate(fact, "Country").sort_values(
        ["RevenuePence", "Country"], ascending=[False, True]
    )
    country["RevenueShare"] = country.RevenuePence / fact.RevenuePence.sum()
    products = aggregate(fact, "StockCode").sort_values(
        ["RevenuePence", "StockCode"], ascending=[False, True]
    )
    names = fact.sort_values(["InvoiceDate", "SourceRow"]).drop_duplicates(
        "StockCode", keep="last"
    )[["StockCode", "Description"]]
    products = products.merge(names, on="StockCode", validate="one_to_one")
    segment = (
        customers.groupby("Segment")
        .agg(Customers=("CustomerID", "size"), RevenuePence=("MonetaryPence", "sum"))
        .reset_index()
    )
    segment["Revenue"] = segment.RevenuePence / 100
    segment["RevenueShare"] = segment.RevenuePence / fact.RevenuePence.sum()
    calendar = pd.DataFrame(
        {"Date": pd.date_range(f"{fact.Year.min()}-01-01", f"{fact.Year.max()}-12-31")}
    )
    calendar["Year"] = calendar.Date.dt.year
    calendar["MonthNumber"] = calendar.Date.dt.month
    calendar["MonthName"] = calendar.Date.dt.month_name()
    calendar["YearMonth"] = calendar.Date.dt.strftime("%Y-%m")
    calendar["YearMonthNumber"] = calendar.Year * 100 + calendar.MonthNumber
    calendar["WeekdayNumber"] = calendar.Date.dt.dayofweek + 1
    calendar["DayOfWeek"] = calendar.Date.dt.day_name()
    calendar["IsFullMonth"] = (
        calendar.Date.dt.to_period("M").dt.start_time.ge(
            pd.Timestamp(fact.InvoiceDay.min())
        )
    ) & (
        calendar.Date.dt.to_period("M")
        .dt.end_time.dt.normalize()
        .le(pd.Timestamp(fact.InvoiceDay.max()))
    )
    activity = fact[["CustomerID", "InvoiceDate"]].copy()
    activity["ActivityMonth"] = activity.InvoiceDate.dt.to_period("M")
    activity["CohortMonth"] = activity.groupby("CustomerID").ActivityMonth.transform(
        "min"
    )
    activity["MonthIndex"] = activity.ActivityMonth.astype(
        "int64"
    ) - activity.CohortMonth.astype("int64")
    cohorts = (
        activity.groupby(["CohortMonth", "ActivityMonth", "MonthIndex"])
        .CustomerID.nunique()
        .rename("ActiveCustomers")
        .reset_index()
    )
    sizes = cohorts.loc[
        cohorts.MonthIndex.eq(0), ["CohortMonth", "ActiveCustomers"]
    ].rename(columns={"ActiveCustomers": "CohortSize"})
    panel = pd.DataFrame(
        [
            (cohort, month, month.ordinal - cohort.ordinal)
            for cohort in sizes.CohortMonth
            for month in pd.period_range(cohort, activity.ActivityMonth.max(), freq="M")
        ],
        columns=["CohortMonth", "ActivityMonth", "MonthIndex"],
    )
    cohorts = panel.merge(
        cohorts, on=["CohortMonth", "ActivityMonth", "MonthIndex"], how="left"
    )
    cohorts["ActiveCustomers"] = cohorts.ActiveCustomers.fillna(0).astype(int)
    cohorts = cohorts.merge(sizes, on="CohortMonth", validate="many_to_one")
    cohorts["RetentionRate"] = cohorts.ActiveCustomers / cohorts.CohortSize
    for col in ["CohortMonth", "ActivityMonth"]:
        cohorts[col] = cohorts[col].astype(str)
    return {
        "orders": orders,
        "customer_metrics": customers,
        "calendar": calendar,
        "monthly_revenue": monthly,
        "country_revenue": country,
        "product_performance": products,
        "daily_revenue": aggregate(fact, ["WeekdayNumber", "DayOfWeek"]),
        "hourly_revenue": aggregate(fact, "Hour"),
        "rfm_segments": segment,
        "cohort_retention": cohorts,
    }

## 6. Check small boundary cases before the full run

These readable checks replace a separate tests folder. They check cancellation/duplicate/missing-customer precedence, half-penny rounding, order-level AOV, conflicting order ownership and tie-preserving scores.

In [ ]:
def example(invoice="100", customer="1", quantity="1", price="0.005", stock="SKU1"):
    return dict(zip(SOURCE_COLUMNS, [invoice, stock, "Example", quantity, "2010-01-01 10:00:00", price, customer, "UK"]))

sample = pd.DataFrame([example(), example(), example("C101"), example("102", customer=None), example("103", quantity="15", price="0.001")])
checked, audit, excluded = clean(sample)
assert checked.SourceRow.tolist() == [1, 5]
assert checked.RevenuePence.tolist() == [1, 2]
assert audit["exclusions"] == {"cancelled_invoice": 1, "exact_duplicate": 1, "missing_customer": 1}
assert len(sample) == len(checked) + len(excluded)
orders_example, _, _ = clean(pd.DataFrame([example(price="10"), example(price="20", stock="SKU2"), example("101", price="30")]))
assert kpis(orders_example)["average_order_value_gbp"] == 30
try:
    clean(pd.DataFrame([example(), example(customer="2")]))
except ValueError as error:
    assert "multiple customers" in str(error)
else:
    raise AssertionError("Conflicting invoice customers were accepted")
values = pd.Series([1, 1, 3, 4, 4], index=list("abcde"))
assert score(values).a == score(values).b
pd.testing.assert_series_equal(score(values).sort_index(), score(values.sample(frac=1, random_state=4)).sort_index())
print("Cleaning, money, order-grain and scoring boundary checks passed.")

## 7. Write the checked fact and audit

The fact is under data/processed; rejected row numbers and mutually exclusive reasons are under data/quality. Never force the expected baseline when changing the source.

In [ ]:
def run_clean(source: Path = RAW) -> tuple[pd.DataFrame, dict]:
    raw = load_raw(source)
    fact, quality, rejected = clean(raw)
    with Path(source).open("rb") as stream:
        quality["source_sha256"] = hashlib.file_digest(stream, "sha256").hexdigest()
    quality["source_file"] = Path(source).name
    PROCESSED.mkdir(parents=True, exist_ok=True)
    fact.to_csv(
        PROCESSED / "cleaned_retail.csv", index=False, date_format="%Y-%m-%d %H:%M:%S"
    )
    QUALITY.mkdir(parents=True, exist_ok=True)
    rejected.to_csv(QUALITY / "excluded_rows.csv", index=False)
    write_json(QUALITY / "cleaning_summary.json", quality)
    write_json(QUALITY / "kpis.json", kpis(fact))
    print(
        f"Cleaned {quality['source_rows']:,} source rows -> {len(fact):,} eligible sales lines."
    )
    return fact, quality

def run_analysis(fact: pd.DataFrame) -> dict[str, pd.DataFrame]:
    tables = build_tables(fact)
    TABLES.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        destination = (
            PROCESSED if name in {"orders", "customer_metrics", "calendar"} else TABLES
        )
        table.to_csv(
            destination / f"{name}.csv", index=False, date_format="%Y-%m-%d %H:%M:%S"
        )
    print(f"Exported {len(tables)} analytical tables.")
    return tables

In [ ]:
fact, quality = run_clean(SOURCE)
tables = run_analysis(fact)
display(pd.Series(kpis(fact), name="Measured value").to_frame())
display(pd.Series(quality["exclusions"], name="Excluded lines").to_frame())
assert quality["source_rows"] == len(fact) + quality["excluded_rows"]
assert fact.SourceRow.is_unique
print("Customers recorded in multiple countries:", quality["customer_multiple_countries"])

## Explain before continuing

Why does repeat-row removal sometimes change sales without changing order counts? What is the source of every generated column? Why are RFM groups fixed at the historical cutoff? Open power_bi/tables and data/quality, then continue to 03.